In [ ]:
import pandas as pd
import numpy as np

In [ ]:
cl1=pd.read_excel('/content/Claim_Lines_2026_01_Cleaned (1).xlsx')
cl2=pd.read_excel('/content/Claim_Lines_2026_02_Cleaned.xlsx')
cl3=pd.read_excel('/content/Claim_Lines_2026_03_Cleaned.xlsx')

In [ ]:
ch1=pd.read_excel('/content/Claim_header_2026_01_Cleaned.xlsx')
ch2=pd.read_excel('/content/Claim_header_2026_02_Cleaned.xlsx')
ch3=pd.read_excel('/content/Claim_header_2026_03_Cleaned.xlsx')

In [ ]:
print(cl1.shape)
print(cl2.shape)
print(cl3.shape)

(12914, 11)
(9799, 11)
(9052, 11)


In [ ]:
print(ch1.shape)
print(ch2.shape)
print(ch3.shape)

(6404, 18)
(5266, 18)
(4937, 19)


In [ ]:
master_cl = pd.concat(
    [cl1, cl2, cl3],
    ignore_index=True
)

print("Master Claim Lines:", master_cl.shape)

Master Claim Lines: (31765, 11)


In [ ]:
master_ch = pd.concat(
    [ch1, ch2, ch3],
    ignore_index=True
)

print("Master Claim Header:", master_ch.shape)

Master Claim Header: (16607, 19)


In [ ]:
provider_data=pd.merge(
    master_cl,
    master_ch[['claim_id','provider_id']],
    on='claim_id',
    how='left'
)

In [ ]:
provider_summary = provider_data.groupby('provider_id').agg(
    total_claims=('claim_id', 'nunique'),
    total_billed=('billed_amount', 'sum'),
    total_paid=('paid_amount', 'sum'),
    billed_anomalies=('billed_anomaly', 'sum'),
    allowed_anomalies=('allowed_anomaly', 'sum'),
    paid_anomalies=('paid_anomaly', 'sum')
).reset_index()

In [ ]:
provider_summary['total_anomalies'] = (
    provider_summary['billed_anomalies'] +
    provider_summary['allowed_anomalies'] +
    provider_summary['paid_anomalies']
)

provider_summary['anomaly_rate'] = (
    provider_summary['total_anomalies'] /
    (provider_summary['total_claims'] * 3)
) * 100

In [ ]:
provider_summary['risk_score'] = (
    provider_summary['anomaly_rate'].clip(0, 100)
)

In [ ]:
def risk_category(score):
    if score >= 60:
        return 'High Risk'
    elif score >= 30:
        return 'Medium Risk'
    else:
        return 'Low Risk'

provider_summary['risk_category'] = (
    provider_summary['risk_score'].apply(risk_category)
)

In [ ]:
provider_summary.sort_values(
    'risk_score',
    ascending=False
).head(20)

,provider_id,total_claims,total_billed,total_paid,billed_anomalies,allowed_anomalies,paid_anomalies,total_anomalies,anomaly_rate,risk_score,risk_category
31,P2032,14,843686.90,283964.51,17,13,13,43,102.380952,100.000000,High Risk
90,P2092,24,2083514.87,592882.97,73,60,61,194,269.444444,100.000000,High Risk
44,P2045,39,3434714.77,171175.13,145,28,18,191,163.247863,100.000000,High Risk
45,P2046,27,3106718.77,132928.70,127,20,16,163,201.234568,100.000000,High Risk
77,P2079,5,238030.32,99119.32,5,5,5,15,100.000000,100.000000,High Risk
89,P2091,33,2203850.46,779175.80,101,78,84,263,265.656566,100.000000,High Risk
181,P2184,20,1595970.03,472272.23,64,54,59,177,295.000000,100.000000,High Risk
180,P2183,17,1208595.58,325369.26,64,35,41,140,274.509804,100.000000,High Risk
212,P2216,2,106016.52,40086.06,2,2,2,6,100.000000,100.000000,High Risk
213,P2217,3,154339.68,51564.27,3,3,3,9,100.000000,100.000000,High Risk


In [ ]:
print(master_cl.columns.tolist())
print(master_ch.columns.tolist())

['claim_id', 'line_number', 'procedure_code', 'units', 'billed_amount', 'allowed_amount', 'paid_amount', 'line_status', 'billed_anomaly', 'allowed_anomaly', 'paid_anomaly']
['claim_id', 'member_id', 'provider_id', 'claim_type', 'place_of_service', 'service_from_date', 'service_to_date', 'received_date', 'primary_diagnosis', 'total_billed', 'total_allowed', 'total_paid', 'member_responsibility', 'claim_status', 'submission_channel', 'total_billed_anomoly', 'total_allowed_anomaly', 'total_paid_anomaly', 'telehealth_flag']


In [ ]:
provider_summary.to_excel('Provider_Risk_Score_Analysis.xlsx', index=False)

In [ ]:
from google.colab import files
files.download('Provider_Risk_Score_Analysis.xlsx')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>